In [1]:
from sampler import ZeoSampler

ModuleNotFoundError: No module named 'sampler'

In [2]:
from torch.utils.data import BatchSampler
from torch.utils.data import Dataset
from numpy.random import choice, shuffle
import numpy as np

In [3]:
class ZeoSampler(BatchSampler):
    def __init__(self, zeolite_codes: list, batch_size, num_samples=1500):
        self.batch_size = batch_size
        self.num_samples = num_samples

        self.zeolite_codes = zeolite_codes
        self.unique_zeo_codes = set(zeolite_codes)
        self.unique_zeo_codes_num = len(self.unique_zeo_codes)
        self.most_zeo_code = max([zeolite_codes.count(zeo_code) for zeo_code in self.unique_zeo_codes])

    def __iter__(self):
        batch_indices = []

        # Sample indices for each zeolite code
        for zeo_code in self.unique_zeo_codes:
            zeo_code_indices = [idx for idx, value in enumerate(self.zeolite_codes) if zeo_code == value]
            repeats = np.floor(self.most_zeo_code // len(zeo_code_indices)).astype(int)
            n_to_sample = self.most_zeo_code % repeats
            zeo_code_indices = np.tile(zeo_code_indices, repeats).tolist()

            if n_to_sample > 0:
                zeo_code_indices.extend(np.random.choice(zeo_code_indices, size=n_to_sample, replace=False).tolist())

            batch_indices.extend(zeo_code_indices)

        # Shuffle and pad indices to fit batches
        np.random.shuffle(batch_indices)
        indices_to_add = self.batch_size - (len(batch_indices) % self.batch_size)
        if indices_to_add < self.batch_size:
            batch_indices.extend([-1] * indices_to_add)

        # Yield one batch at a time
        for i in range(0, len(batch_indices), self.batch_size):
            batch = batch_indices[i:i + self.batch_size]
            # print([idx for idx in batch if idx != -1])
            yield [idx for idx in batch if idx != -1]  # Remove padded indices

    def __len__(self):
        return int(np.ceil(len(self.unique_zeo_codes) * self.most_zeo_code / self.batch_size))

In [4]:
class TestDataset(Dataset):
    def __init__(self, graphs):
        self.graphs = graphs

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        print(index)
        return Data(
            x=torch.randn(5,3)
        )

In [5]:
import torch
from torch_geometric.loader import DataLoader 
from torch_geometric.data import Data

X = 5 * ['MOR'] + 2 * ["TON"] + 1 * ['MFI']
graphs = [Data(x=torch.randn(5,3)) for i in range(8)]

sampler = ZeoSampler(X, batch_size=4)
dataset = TestDataset(graphs=graphs)
data_loader = DataLoader(dataset=dataset, batch_sampler=sampler)

In [6]:
out = list(sampler)

out2 = []

for o in out:
    out2.extend(o)

sorted(out2)

[0, 1, 2, 3, 4, 5, 5, 6, 6, 6, 7, 7, 7, 7, 7]

In [6]:
print(len(data_loader))
for data in data_loader:
    print(data)

4
7
7
2
5
DataBatch(x=[20, 3], batch=[20], ptr=[5])
4
7
6
5
DataBatch(x=[20, 3], batch=[20], ptr=[5])
0
6
7
3
DataBatch(x=[20, 3], batch=[20], ptr=[5])
1
7
5
DataBatch(x=[15, 3], batch=[15], ptr=[4])
